# 합성 학습 쿼리 생성 — A 업종·C 상황 (Colab GPU)

코퍼스 가맹점마다 오픈 LLM으로 "이 가게를 찾을 때 칠 말"(A)과 "이 가게가 필요한 상황에서 칠 말"(C)을 만듭니다.
계획은 `docs/SERVICE_QUERYSET.md` 4절. 거르기(왕복 일치·cross-encoder)와 정답 확장은 이 다음 단계입니다.

1. 로컬: `python scripts/export_generation_inputs.py` → `colab_upload/generation/`의 `stores.csv`, `fewshot.csv`, `inputs_meta.json`을
   Drive `내 드라이브/store-search-ai/generation/`에 올림
2. Colab: 런타임 유형 **GPU(L4 또는 A100 권장, 메모리 20GB 미만이면 작은 모델로 자동 전환)** → `SMOKE=True`로 끝까지 한 번 →
   출력 예시를 보고 → `SMOKE=False`로 전체 실행
   - 중간에 끊겨도 다시 실행하면 이미 만든 가맹점은 건너뜁니다.
   - 첫 셀이 vllm을 설치한 뒤 런타임을 **자동 재시작**합니다("세션이 다운됨" 알림은 정상). 그다음 첫 셀부터 다시 실행하면 설치를 건너뜁니다.
     설치 중 나오는 `pip's dependency resolver ...` 경고(cudf·cuml·gradio·google-adk·numba-cuda)는 이 노트북과 무관합니다.
   - T4에서 vLLM 엔진 시작이 실패하면 transformers로 자동 전환합니다(`BACKEND =` 출력 확인). 결과는 같은 형식이고 속도만 느립니다.
3. 결과: 같은 폴더의 `generated_A_<버전>.csv`, `generated_C_<버전>.csv`를 내려받아 로컬 `data/finetune/synthetic/`에 넣음

프롬프트·파싱(`# @cell prompts`)을 고치면 `PROMPT_VERSION`을 올리세요. 버전마다 결과 파일이 따로 생겨 섞이지 않습니다.
`tests/test_generate_queries_notebook.py`가 이 셀을 실행해 확인합니다.

In [ ]:
# 설치 — torch를 import하기 전에 맨 먼저 실행한다(vllm이 torch를 바꿀 수 있어서, 설치 후 런타임을 재시작해야 한다).
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("vllm") is None:
    # requests는 google-colab이 요구하는 버전으로 고정(Drive 마운트). 그 밖에 cudf·cuml·gradio·google-adk·numba-cuda
    # 버전 충돌 경고가 나와도 이 노트북은 그 패키지들을 쓰지 않으므로 무시해도 된다.
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm", "requests==2.32.4"],
                            capture_output=True, text=True)
    print(result.stdout[-2000:], result.stderr[-3000:])
    if result.returncode != 0:
        raise RuntimeError("vllm 설치 실패 — 위 메시지 확인")
    print("vllm 설치 완료 → 런타임을 재시작합니다. 재시작되면 이 셀부터 다시 실행하세요(설치는 건너뜁니다).")
    os.kill(os.getpid(), 9)
else:
    import vllm
    print("vllm", vllm.__version__, "— 설치 건너뜀")

In [ ]:
import torch
assert torch.cuda.is_available(), "런타임 > 런타임 유형 변경 > GPU를 선택하세요"
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
GPU_CC = torch.cuda.get_device_capability(0)
print(GPU_NAME, f"{GPU_MEM_GB:.0f}GB", "compute capability", GPU_CC)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

GEN_DIR = Path("/content/drive/MyDrive/store-search-ai/generation")
SMOKE = True               # True: 가맹점 20곳만(출력 파일 이름에 _smoke)
BUNDLES = ["A", "C"]
PROMPT_VERSION = "v1"
# 8B는 bf16 가중치만 약 16GB — L4(24GB)·A100에서. 그보다 작은 GPU(T4 16GB)면 4B로 바꾼다.
MODEL = "Qwen/Qwen3-8B" if GPU_MEM_GB >= 20 else "Qwen/Qwen3-4B-Instruct-2507"
TEMPERATURE = 0.8
MAX_NEW_TOKENS = 160       # 검색어 3개 JSON에 충분
CHUNK = 500                # 이만큼 생성할 때마다 Drive에 저장(끊겨도 이어서)
USE_VLLM = True            # vLLM 시작이 실패하면 자동으로 transformers로 바꾼다. 처음부터 transformers로 하려면 False
HF_BATCH = 32              # transformers 생성 배치(T4 16GB + 4B 기준)
print("MODEL =", MODEL)

In [ ]:
# @cell prompts
import hashlib
import json
import random
import re

SYSTEM = "당신은 한국 지도 앱의 실제 검색 로그처럼 짧고 자연스러운 검색어를 만드는 데이터 작성자입니다."

RULES = {
    "A": (
        "온누리상품권 가맹점 하나를 보고, 이 가게를 찾는 사람이 지도 앱 검색창에 칠 법한 검색어를 {n}개 만드세요.\n\n"
        "규칙\n"
        "- 가게 이름(상호)은 쓰지 않습니다. 업종·품목·서비스로 찾는 검색어입니다.\n"
        "- 서로 다른 방식으로 씁니다: 같은 뜻의 다른 말, 더 넓거나 좁은 품목, 짧은 구어체(\"~ 파는 곳\", \"~ 할 데\").\n"
        "- 실제로 칠 만큼 짧게(2~15자), 오타 없이.\n"
        "- 이 가게가 실제로 파는 것·하는 일에서 벗어나지 않습니다."
    ),
    "C": (
        "온누리상품권 가맹점 하나를 보고, 이 가게가 필요해지는 생활 상황을 떠올린 뒤 그 상황의 사람이 지도 앱 검색창에 칠 법한 "
        "검색어를 {n}개 만드세요.\n\n"
        "규칙\n"
        "- 가게 이름은 쓰지 않습니다. 취급품목 단어를 그대로 반복하기보다 목적·하려는 일·필요한 물건으로 씁니다.\n"
        "- 상황은 서로 다르게, 실제로 칠 만큼 짧게(2~20자), 오타 없이.\n"
        "- 이 가게에서 실제로 해결되는 상황만 씁니다."
    ),
}


def pick_examples(fewshot, bundle: str, doc_id: str, k: int = 6) -> list[str]:
    """가맹점마다 다른 업종 예시 k개(doc_id로 seed를 고정해 다시 돌려도 같은 프롬프트)."""

    rows = fewshot[fewshot["bundle"] == bundle]
    by_family = rows.groupby("family")["query"].apply(list)
    rng = random.Random(int(hashlib.md5(f"{bundle}:{doc_id}".encode()).hexdigest(), 16))
    families = rng.sample(sorted(by_family.index), min(k, len(by_family)))
    return [f"- {fam}: " + " / ".join(by_family[fam][:3]) for fam in families]


def build_messages(bundle: str, store: dict, examples: list[str], n: int) -> list[dict]:
    user = (
        RULES[bundle].format(n=n)
        + "\n\n다른 업종의 예시\n" + "\n".join(examples)
        + f"\n\n가게\n- 상호: {store['store_name']}\n- 취급품목: {store['item_text'] or '(없음)'}\n- 시장유형: {store['market_type']}"
        + '\n\nJSON 배열로만 답하세요. 예: ["검색어1", "검색어2"]'
    )
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]


def parse_queries(text: str, store_name: str, max_len: int = 25) -> list[str]:
    """모델 출력 → 검색어 목록. JSON 배열을 먼저 찾고, 없으면 줄 단위로 읽는다.
    빈 것·너무 긴 것·중복·상호가 들어간 것은 뺀다."""

    text = re.sub(r"<think>.*?</think>", "", text or "", flags=re.S)
    items = None
    match = re.search(r"\[.*?\]", text, flags=re.S)
    if match:
        try:
            items = [str(x) for x in json.loads(match.group(0))]
        except json.JSONDecodeError:
            items = None
    if items is None:
        items = [re.sub(r'^[\s\-\*\d\.\)"]+|["\s]+$', "", line) for line in text.splitlines()]
    name_key = re.sub(r"\s+", "", re.sub(r"\(.*?\)", "", store_name or "")).casefold()
    out, seen = [], set()
    for q in items:
        q = re.sub(r"\s+", " ", q).strip().strip('"').strip()
        key = q.replace(" ", "").casefold()
        if not (2 <= len(q) <= max_len) or key in seen:
            continue
        if len(name_key) >= 2 and (name_key in key or key in name_key):
            continue
        seen.add(key)
        out.append(q)
    return out

In [ ]:
import pandas as pd

stores = pd.read_csv(GEN_DIR / "stores.csv", encoding="utf-8-sig", dtype=str).fillna("")
fewshot = pd.read_csv(GEN_DIR / "fewshot.csv", encoding="utf-8-sig")
meta = json.loads((GEN_DIR / "inputs_meta.json").read_text(encoding="utf-8"))
N_PER_STORE = meta["queries_per_store"]
if SMOKE:
    stores = stores.head(20)
print(f"가맹점 {len(stores)}곳 × {N_PER_STORE}개 × {BUNDLES}, 예시 {meta['fewshot_rows']}")
first = stores.iloc[0].to_dict()
print("\n".join(m["content"] for m in build_messages("A", first, pick_examples(fewshot, "A", first["doc_id"]), N_PER_STORE)))

In [ ]:
import gc

from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
if tok.pad_token is None:
    tok.pad_token = tok.eos_token


def to_prompt(messages):
    # Qwen3는 기본이 '생각' 모드라 끈다(다른 모델은 이 인자를 무시)
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)


generate_texts = None
if USE_VLLM:
    try:
        from vllm import LLM, SamplingParams

        llm = LLM(
            model=MODEL,
            dtype="half" if GPU_CC[0] < 8 else "auto",   # T4(7.5)는 bf16 미지원
            max_model_len=2048,
            gpu_memory_utilization=0.85,
            seed=20261019,
            enforce_eager=True,   # torch.compile·CUDA graph를 끈다 — T4에서 엔진 시작 실패를 줄이고 시작도 빠르다
        )
        params = SamplingParams(temperature=TEMPERATURE, top_p=0.95, max_tokens=MAX_NEW_TOKENS)

        def generate_texts(prompts):
            return [o.outputs[0].text for o in llm.generate(prompts, params)]

        BACKEND = "vllm"
    except Exception as e:   # 엔진 시작 실패(주로 T4) → transformers로
        print("vLLM 시작 실패 → transformers로 생성합니다(더 느림):", repr(e)[:300])
        gc.collect()
        torch.cuda.empty_cache()

if generate_texts is None:
    from transformers import AutoModelForCausalLM

    model = AutoModelForCausalLM.from_pretrained(
        MODEL, dtype=torch.float16 if GPU_CC[0] < 8 else torch.bfloat16
    ).to("cuda").eval()
    torch.manual_seed(20261019)

    def generate_texts(prompts):
        out = []
        for i in range(0, len(prompts), HF_BATCH):
            batch = tok(prompts[i:i + HF_BATCH], return_tensors="pt", padding=True).to("cuda")
            with torch.no_grad():
                gen = model.generate(**batch, do_sample=True, temperature=TEMPERATURE, top_p=0.95,
                                     max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tok.pad_token_id)
            out += tok.batch_decode(gen[:, batch["input_ids"].shape[1]:], skip_special_tokens=True)
        return out

    BACKEND = "transformers"

print("BACKEND =", BACKEND)

In [ ]:
suffix = "_smoke" if SMOKE else ""
for bundle in BUNDLES:
    out_path = GEN_DIR / f"generated_{bundle}_{PROMPT_VERSION}{suffix}.csv"
    done = set(pd.read_csv(out_path, encoding="utf-8-sig", usecols=["doc_id"])["doc_id"]) if out_path.exists() else set()
    todo = stores[~stores["doc_id"].isin(done)]
    print(f"[{bundle}] 이미 {len(done)}곳, 남은 {len(todo)}곳 → {out_path.name}")
    for start in range(0, len(todo), CHUNK):
        chunk = todo.iloc[start:start + CHUNK].to_dict("records")
        prompts = [to_prompt(build_messages(bundle, row, pick_examples(fewshot, bundle, row["doc_id"]), N_PER_STORE))
                   for row in chunk]
        rows = []
        for row, raw in zip(chunk, generate_texts(prompts)):
            queries = parse_queries(raw, row["store_name"])
            if not queries:   # 실패도 기록해야 다시 돌릴 때 건너뛰고 실패율을 볼 수 있다
                rows.append({"doc_id": row["doc_id"], "bundle": bundle, "rank": -1, "query": "", "raw": raw[:300]})
            rows += [{"doc_id": row["doc_id"], "bundle": bundle, "rank": i, "query": q, "raw": ""}
                     for i, q in enumerate(queries)]
        frame = pd.DataFrame(rows).assign(model=MODEL, backend=BACKEND, prompt_version=PROMPT_VERSION)
        frame.to_csv(out_path, mode="a", header=not out_path.exists(), index=False, encoding="utf-8-sig",
                     lineterminator="\n")
        print(f"  {min(start + CHUNK, len(todo))}/{len(todo)} 저장")

In [ ]:
for bundle in BUNDLES:
    df = pd.read_csv(GEN_DIR / f"generated_{bundle}_{PROMPT_VERSION}{suffix}.csv", encoding="utf-8-sig")
    ok = df[df["rank"] >= 0]
    print(f"[{bundle}] 가맹점 {df['doc_id'].nunique()}곳, 쿼리 {len(ok)}개, 파싱 실패 {(df['rank'] < 0).sum()}곳, "
          f"고유 쿼리 {ok['query'].nunique()}개")
    sample = ok.merge(stores, on="doc_id").sample(min(15, len(ok)), random_state=0)
    print(sample[["store_name", "item_text", "query"]].to_string(index=False))